# Ain Shams University - Faculty of Engineering
### CSE485 / CSE488: Deep Learning (Fall 2026)
## Assignment 1: MicroGrad (Scalar Autograd Engine)
**Student Name:** Madonna Emad Labib  
**Student ID:** 23P0325  
**Submission:** Part B - Demo Jupyter Notebook Pipeline
***
This notebook demonstrates:
1. Scalar automatic differentiation and analytical verification.
2. Computational graph visualization using Graphviz DAG tracing.
3. Neural Network API (`Neuron`, `Layer`, `MLP`).
4. End-to-end training of an MLP on a non-linear 2D classification dataset.
5. Loss curves and decision boundary contour visualization.


In [ ]:
import sys
import os
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from graphviz import Digraph

# Ensure local source package is discoverable
sys.path.append(os.path.abspath('.'))
from src.engine import Value
from src.nn import Neuron, Layer, MLP

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
print("Modules imported successfully!")


## 1. Scalar Autograd Verification
We construct a composite computation graph:
$$f(a, b) = (a \cdot b) \cdot (a + b) = a^2 b + a b^2$$
Analytically:
$$\frac{\partial f}{\partial a} = 2ab + b^2, \quad \frac{\partial f}{\partial b} = a^2 + 2ab$$
For $a = -2.0, b = 3.0$:
$$\frac{\partial f}{\partial a} = 2(-2)(3) + (3)^2 = -12 + 9 = -3.0$$
$$\frac{\partial f}{\partial b} = (-2)^2 + 2(-2)(3) = 4 - 12 = -8.0$$


In [ ]:
# Create input nodes
a = Value(-2.0, label='a')
b = Value(3.0, label='b')

# Forward pass
d = a * b; d.label = 'd'
e = a + b; e.label = 'e'
f = d * e; f.label = 'f'

# Backward pass
f.backward()

print(f"Forward Output f = {f.data:.4f} (Expected: {-30.0})")
print(f"Gradient df/da   = {a.grad:.4f} (Expected: {-3.0})")
print(f"Gradient df/db   = {b.grad:.4f} (Expected: {-8.0})")

assert f.data == -30.0
assert a.grad == -3.0
assert b.grad == -8.0
print("✓ Exact analytical gradient match verified!")


## 2. Computational Graph Visualization
We build a DAG visualizer that traverses node parents and renders values, gradients, and operation nodes using Graphviz.


In [ ]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        lbl_str = f" {n.label} |" if n.label else ""
        dot.node(name=uid, label=f"{{{lbl_str} data {n.data:.4f} | grad {n.grad:.4f}}}", shape='record')
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

dot = draw_dot(f)
dot


## 3. The Diamond Graph Trap & Gradient Accumulation
When a node branches to multiple children and rejoins:
$$c = a + a = 2a, \quad d = c \cdot a = 2a^2$$
$$\frac{\partial d}{\partial a} = 4a$$
For $a = 3.0$, $\frac{\partial d}{\partial a} = 12.0$.
If gradients were assigned with `=` instead of accumulated with `+=`, gradients would be erroneously overwritten.


In [ ]:
a = Value(3.0, label='a')
c = a + a; c.label = 'c'
d = c * a; d.label = 'd'
d.backward()

print(f"Output d.data = {d.data:.4f}")
print(f"Gradient da   = {a.grad:.4f} (Expected: 12.0)")
assert a.grad == 12.0
print("✓ Gradient accumulation across diamond graph verified!")


## 4. Synthetic Non-Linear 2D Classification Dataset
We generate a concentric 2-circle dataset that cannot be separated linearly.


In [ ]:
def make_circles(n_samples=100, noise=0.08, factor=0.5):
    n_samples_out = n_samples // 2
    n_samples_in = n_samples - n_samples_out
    
    # Outer circle
    linspace_out = np.linspace(0, 2 * np.pi, n_samples_out, endpoint=False)
    outer_circ_x = np.cos(linspace_out) + np.random.normal(0, noise, n_samples_out)
    outer_circ_y = np.sin(linspace_out) + np.random.normal(0, noise, n_samples_out)
    
    # Inner circle
    linspace_in = np.linspace(0, 2 * np.pi, n_samples_in, endpoint=False)
    inner_circ_x = factor * np.cos(linspace_in) + np.random.normal(0, noise, n_samples_in)
    inner_circ_y = factor * np.sin(linspace_in) + np.random.normal(0, noise, n_samples_in)
    
    X = np.vstack([np.column_stack([outer_circ_x, outer_circ_y]),
                   np.column_stack([inner_circ_x, inner_circ_y])])
    # Labels: -1 for outer circle, +1 for inner circle
    y = np.hstack([-np.ones(n_samples_out), np.ones(n_samples_in)])
    return X, y

X, y = make_circles(n_samples=100, noise=0.08, factor=0.45)

plt.figure(figsize=(6, 6))
plt.scatter(X[y == 1, 0], X[y == 1, 1], c='crimson', label='Class +1 (Inner)', edgecolors='k')
plt.scatter(X[y == -1, 0], X[y == -1, 1], c='royalblue', label='Class -1 (Outer)', edgecolors='k')
plt.title("Synthetic Non-Linear 2D Dataset")
plt.xlabel("Feature $x_1$")
plt.ylabel("Feature $x_2$")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


## 5. Multi-Layer Perceptron (MLP) Architecture & Training
We initialize a network with architecture:
$$\text{Input (2)} \to \text{Hidden (8, tanh)} \to \text{Hidden (8, tanh)} \to \text{Output (1, tanh)}$$
We optimize using max-margin hinge loss with L2 weight decay:
$$\mathcal{L} = \frac{1}{N} \sum_{i=1}^N \max(0, 1 - y_i \cdot \hat{y}_i) + \alpha \sum_{w} w^2$$


In [ ]:
# Initialize model
model = MLP(nin=2, nouts=[8, 8, 1])
print("Model Architecture:", model)
print("Total Trainable Parameters:", len(model.parameters()))

# Training hyperparameters
learning_rate = 0.1
epochs = 60
alpha = 1e-4  # L2 regularization strength
losses = []
accuracies = []

for k in range(epochs):
    # Forward pass
    total_loss = Value(0.0)
    correct = 0
    
    for xi, yi in zip(X, y):
        pred = model(xi)
        # Margin loss: max(0, 1 - yi * pred)
        margin = Value(1.0) + (pred * (-float(yi)))
        loss_i = margin if margin.data > 0 else Value(0.0)
        total_loss = total_loss + loss_i
        
        # Accuracy tracking
        if (pred.data > 0) == (yi > 0):
            correct += 1
            
    # Add L2 regularization
    reg_loss = alpha * sum((p * p for p in model.parameters()), Value(0.0))
    loss = (total_loss * (1.0 / len(X))) + reg_loss
    
    # Backward pass
    model.zero_grad()
    loss.backward()
    
    # Gradient descent update with learning rate decay
    lr = learning_rate * (1.0 - 0.7 * (k / epochs))
    for p in model.parameters():
        p.data -= lr * p.grad
        
    acc = correct / len(X) * 100.0
    losses.append(loss.data)
    accuracies.append(acc)
    
    if (k + 1) % 10 == 0 or k == 0:
        print(f"Epoch {k+1:02d}/{epochs} | Loss: {loss.data:.4f} | Accuracy: {acc:.1f}%")

print(f"Final Convergence - Loss: {losses[-1]:.4f} | Accuracy: {accuracies[-1]:.1f}%")


## 6. Training Dynamics & Decision Boundary Visualization


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss & Accuracy Curves
ax1.plot(losses, color='crimson', lw=2, label='Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss', color='crimson')
ax1.tick_params(axis='y', labelcolor='crimson')
ax1.grid(True, linestyle='--', alpha=0.5)

ax1_acc = ax1.twinx()
ax1_acc.plot(accuracies, color='teal', lw=2, linestyle='--', label='Accuracy (%)')
ax1_acc.set_ylabel('Accuracy (%)', color='teal')
ax1_acc.tick_params(axis='y', labelcolor='teal')
ax1.set_title('Loss & Accuracy Progression')

# Plot 2: Decision Boundary Contour
h = 0.05
x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

grid_points = np.c_[xx.ravel(), yy.ravel()]
preds = np.array([model([x0, x1]).data for x0, x1 in grid_points])
Z = preds.reshape(xx.shape)

ax2.contourf(xx, yy, Z, levels=20, cmap='coolwarm', alpha=0.7)
ax2.contour(xx, yy, Z, levels=[0.0], colors='black', linewidths=2)
ax2.scatter(X[y == 1, 0], X[y == 1, 1], c='crimson', edgecolors='k', label='Class +1')
ax2.scatter(X[y == -1, 0], X[y == -1, 1], c='royalblue', edgecolors='k', label='Class -1')
ax2.set_title('Learned Non-Linear Decision Boundary (Z=0)')
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')
ax2.legend()
plt.tight_layout()
plt.show()


## 7. Conclusions & Findings
1. **Dynamic DAG Construction**: Dynamic tracing correctly evaluates arbitrary non-linear topologies.
2. **Gradient Accumulation**: Crucial for diamond structures where gradients fan out and fan in.
3. **Capacity of MLP**: An MLP of 2 hidden layers with 8 neurons each easily learns non-linear decision boundaries like concentric circular clusters.
